In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9,4)

In [34]:
patients = pd.read_csv('../data/raw/patients.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/raw/vital_signs.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/raw/clinical_history.csv')
labs = pd.read_csv('../data/raw/laboratory_results.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/raw/sepsis_outcomes.csv', parse_dates=['diagnosis_time'])

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs':labs, 'outcomes': outcomes }
for name, df in tables.items():
    print(f'{name:10s} shape={df.shape}')

patients   shape=(600, 5)
vitals     shape=(11807, 8)
history    shape=(1449, 6)
labs       shape=(2430, 8)
outcomes   shape=(600, 6)


In [35]:
patients.head()

,patient_id,age,gender,medical_conditions,registration_date
0,1,66,Male,"Diabetes, Cancer (active)",2024-10-19
1,2,42,Female,"Diabetes, Chronic Kidney Disease, Immunosuppre...",2025-06-24
2,3,74,Male,Cancer (active),2024-02-28
3,4,77,Female,"Diabetes, Hypertension",2024-07-31
4,5,25,Female,NaN,2024-07-09


In [36]:
vitals.describe().T

,count,mean,min,25%,50%,75%,max,std
observation_id,11807.0,5904.0,1.0,2952.5,5904.0,8855.5,11807.0,3408.531649
patient_id,11807.0,297.125858,1.0,149.0,294.0,444.0,600.0,172.006979
timestamp,11807,2025-01-05 03:10:59.588379648,2024-01-01 04:37:00,2024-07-09 02:07:30,2025-01-15 01:46:00,2025-07-11 20:25:30,2025-12-29 07:13:00,NaN
heart_rate,11490.0,79.892097,43.4,73.2,79.2,85.3,137.4,10.838351
temperature,11514.0,36.84276,34.52,36.55,36.82,37.09,40.25,0.51965
oxygen_saturation,11520.0,97.271389,86.4,96.6,97.4,98.2,100.0,1.555355
respiratory_rate,11519.0,16.543233,8.0,14.6,16.3,18.0,34.6,2.990542
blood_pressure,11545.0,121.214517,55.0,113.8,121.5,129.0,163.9,12.076626


In [37]:
print("Sepsis Prevalance:", outcomes['sepsis_event'].mean().round(3))
outcomes["sepsis_event"].value_counts()

Sepsis Prevalance: 0.12


sepsis_event
False    528
True      72
Name: count, dtype: int64

## Data Quality Asseessment

### Checking for missing rows

In [38]:
def missing_report(df, name):
    miss = df.isna().mean().mul(100).round(2)
    miss = miss[miss > 0]
    if len(miss):
        print(f'--- {name}---')
        print(miss.to_string())

for name, df in tables.items():
    missing_report(df, name)

--- patients---
medical_conditions    25.83
--- vitals---
heart_rate           2.68
temperature          2.48
oxygen_saturation    2.43
respiratory_rate     2.44
blood_pressure       2.22
--- history---
diagnosis_history      2.62
medication_history     9.45
treatment_history     14.56
--- labs---
white_cell_count    2.92
crp                 2.47
lactate             3.05
creatinine          3.42
platelet_count      3.00
--- outcomes---
diagnosis_time    88.0


### Checking for duplicate rows

In [39]:
for name, df in tables.items():
    print(name, 'duplicate rows', df.duplicated().sum())

patients duplicate rows 0
vitals duplicate rows 0
history duplicate rows 0
labs duplicate rows 0
outcomes duplicate rows 0


### Checking for negative values in the numerics

In [40]:
for name, df in tables.items():
    numeric = df.select_dtypes(include=['int64', 'float64'])
    neg = (numeric < 0).sum()
    neg = neg[neg > 0]
    print(f"{name}: {neg.to_string() if len(neg) > 0 else 'No negatives'}")

patients: No negatives
vitals: No negatives
history: No negatives
labs: No negatives
outcomes: No negatives


### Checking for referential integrity

In [41]:
valid_ids = set(patients["patient_id"])
for name, df in [('vitals', vitals), ('history', history), ('labs', labs), ('outcomes', outcomes)]:
    orphans = (~df['patient_id'].isin(valid_ids)).sum()
    print(name, 'orphan patient_id', orphans)

vitals orphan patient_id 0
history orphan patient_id 0
labs orphan patient_id 0
outcomes orphan patient_id 0


In [42]:
ranges = {
    'heart_rate': (30, 220), 'temperature': (32, 43), 'oxygen_saturation': (50, 100),
    'respiratory_rate': (5, 60), 'blood_pressure': (40, 220)

}

for cols, (low, high) in ranges.items():
    bad = ((vitals[cols] < low) | (vitals[cols] > high)).sum()
    print(f'{cols}: {bad} out of range')
 

heart_rate: 0 out of range
temperature: 0 out of range
oxygen_saturation: 0 out of range
respiratory_rate: 0 out of range
blood_pressure: 0 out of range


In [43]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='object')

In [44]:
labs.columns


Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='object')

In [45]:
vital_cols = ['heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'

]

labs_cols = ['white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'

]

vitals[vital_cols] = vitals.groupby('patient_id')[vital_cols].transform(lambda s: s.ffill())
vitals[vital_cols] = vitals[vital_cols].fillna(vitals[vital_cols].median())

labs[labs_cols] = labs.groupby('patient_id')[labs_cols].transform(lambda s: s.ffill())
labs[labs_cols] = labs[labs_cols].fillna(labs[labs_cols].median())

print('remaining missing values -> vitals:', vitals[vital_cols].isna().sum().sum(), '& labs', labs[labs_cols].isna().sum().sum())


remaining missing values -> vitals: 0 & labs 0


In [46]:
patients.to_csv('../data/processed/patients_clean.csv', index=False)
vitals.to_csv('../data/processed/vital_signs_clean.csv', index=False)
history.to_csv('../data/processed/clinical_history_clean.csv', index=False)
labs.to_csv('../data/processed/laboratory_results_clean.csv', index=False)
outcomes.to_csv('../data/processed/sepsis_outcomes_clean.csv', index=False)

# EDA

In [47]:
# Age distribution
print("\n=== Age Distribution ===")
print(patients['age'].describe())
print(f"\nAge categories:")
print(pd.cut(patients['age'], bins=[0,20,40,60,80,100]).value_counts().sort_index())


=== Age Distribution ===
count    600.000000
mean      59.908333
std       17.139122
min       18.000000
25%       48.000000
50%       60.000000
75%       71.000000
max       96.000000
Name: age, dtype: float64

Age categories:
age
(0, 20]        7
(20, 40]      74
(40, 60]     221
(60, 80]     226
(80, 100]     72
Name: count, dtype: int64


In [48]:
# Gender distribution
print("\n=== Gender Distribution ===")
print(patients['gender'].value_counts())
print(patients['gender'].value_counts(normalize=True).mul(100).round(2))


=== Gender Distribution ===
gender
Male                   295
Female                 283
Other/Not specified     22
Name: count, dtype: int64
gender
Male                   49.17
Female                 47.17
Other/Not specified     3.67
Name: proportion, dtype: float64


In [49]:
# Medical conditions
print("\n=== Medical Conditions ===")
if 'medical_conditions' in patients.columns:
    all_conditions = patients['medical_conditions'].str.split(',').explode().str.strip()
    print(all_conditions.value_counts())


=== Medical Conditions ===
medical_conditions
Cancer (active)            92
Chronic Kidney Disease     87
Liver Disease              86
Hypertension               84
COPD                       79
Immunosuppression          77
Coronary Artery Disease    76
Diabetes                   73
Obesity                    66
Name: count, dtype: int64


In [50]:
# Vital signs statistics
print("\n=== Vital Signs Statistics ===")
vitals_cols = ['heart_rate', 'temperature', 'oxygen_saturation', 'respiratory_rate', 'blood_pressure']
existing_vitals = [col for col in vitals_cols if col in vitals.columns]
if existing_vitals:
    print(vitals[existing_vitals].describe().T)



=== Vital Signs Statistics ===
                     count        mean        std    min     25%     50%  \
heart_rate         11807.0   79.897180  10.838750  43.40   73.20   79.20   
temperature        11807.0   36.843379   0.518829  34.52   36.55   36.82   
oxygen_saturation  11807.0   97.269569   1.559637  86.40   96.60   97.40   
respiratory_rate   11807.0   16.537351   2.984104   8.00   14.60   16.30   
blood_pressure     11807.0  121.235496  12.073481  55.00  113.85  121.60   

                      75%     max  
heart_rate          85.30  137.40  
temperature         37.09   40.25  
oxygen_saturation   98.20  100.00  
respiratory_rate    18.00   34.60  
blood_pressure     129.00  163.90  


In [51]:
# Check for abnormal vital signs
print("\n=== Abnormal Vital Signs ===")
if 'heart_rate' in vitals.columns:
    abnormal_hr = vitals[(vitals['heart_rate'] < 60) | (vitals['heart_rate'] > 100)]
    print(f"Abnormal heart rate (<60 or >100): {len(abnormal_hr)} records")

if 'oxygen_saturation' in vitals.columns:
    low_oxygen = vitals[vitals['oxygen_saturation'] < 92]
    print(f"Low oxygen saturation (<92%): {len(low_oxygen)} records")

if 'respiratory_rate' in vitals.columns:
    abnormal_resp = vitals[(vitals['respiratory_rate'] < 12) | (vitals['respiratory_rate'] > 20)]
    print(f"Abnormal respiratory rate (<12 or >20): {len(abnormal_resp)} records ({len(abnormal_resp)/len(vitals)*100:.2f}%)")

if 'temperature' in vitals.columns:
    abnormal_temp = vitals[(vitals['temperature'] < 36.0) | (vitals['temperature'] > 38.0)]
    print(f"Abnormal temperature (<36.0°C or >38.0°C): {len(abnormal_temp)} records ({len(abnormal_temp)/len(vitals)*100:.2f}%)")

if 'blood_pressure' in vitals.columns:
    abnormal_bp = vitals[(vitals['blood_pressure'] < 90) | (vitals['blood_pressure'] > 140)]
    print(f"Abnormal blood pressure (<90 or >140): {len(abnormal_bp)} records ({len(abnormal_bp)/len(vitals)*100:.2f}%)")


=== Abnormal Vital Signs ===
Abnormal heart rate (<60 or >100): 782 records
Low oxygen saturation (<92%): 191 records
Abnormal respiratory rate (<12 or >20): 1408 records (11.93%)
Abnormal temperature (<36.0°C or >38.0°C): 713 records (6.04%)
Abnormal blood pressure (<90 or >140): 789 records (6.68%)


### Laboratory Analysis

In [52]:
print("\n=== Laboratory Results Statistics ===")
lab_cols = ['white_cell_count', 'crp', 'lactate', 'creatinine', 'platelet_count']
existing_labs = [col for col in lab_cols if col in labs.columns]
if existing_labs:
    print(labs[existing_labs].describe().T)

# Check for abnormal lab values
print("\n=== Abnormal Lab Values ===")
if 'white_cell_count' in labs.columns:
    abnormal_wbc = labs[
        (labs['white_cell_count'] < 4) | 
        (labs['white_cell_count'] > 11)
    ]
    print(f"Abnormal WBC (<4 or >11): {len(abnormal_wbc)} records")

if 'crp' in labs.columns:
    high_crp = labs[labs['crp'] > 10]
    print(f"High CRP (>10): {len(high_crp)} records")

if 'lactate' in labs.columns:
    high_lactate = labs[labs['lactate'] > 2.0]
    print(f"High Lactate (>2.0 mmol/L): {len(high_lactate)} records ({len(high_lactate)/len(labs)*100:.2f}%)")

if 'creatinine' in labs.columns:
    high_creatinine = labs[labs['creatinine'] > 1.2]
    print(f"High Creatinine (>1.2 mg/dL): {len(high_creatinine)} records ({len(high_creatinine)/len(labs)*100:.2f}%)")

if 'platelet_count' in labs.columns:
    low_platelets = labs[labs['platelet_count'] < 150]
    print(f"Low Platelet Count (<150): {len(low_platelets)} records ({len(low_platelets)/len(labs)*100:.2f}%)")


=== Laboratory Results Statistics ===
                   count        mean        std   min     25%     50%     75%  \
white_cell_count  2430.0    7.807868   2.410643   1.0    6.33    7.60    8.96   
crp               2430.0   12.207037  25.908068   0.5    3.40    6.30    9.40   
lactate           2430.0    1.139761   0.659189   0.3    0.83    1.03    1.23   
creatinine        2430.0    0.942000   0.284288   0.3    0.76    0.91    1.08   
platelet_count    2430.0  254.253086  45.810641  55.0  224.25  255.00  285.00   

                     max  
white_cell_count   22.15  
crp               193.10  
lactate             5.65  
creatinine          2.38  
platelet_count    388.00  

=== Abnormal Lab Values ===
Abnormal WBC (<4 or >11): 213 records
High CRP (>10): 511 records
High Lactate (>2.0 mmol/L): 129 records (5.31%)
High Creatinine (>1.2 mg/dL): 331 records (13.62%)
Low Platelet Count (<150): 35 records (1.44%)


### Sepsis Outcome Analysis

In [53]:
print("\n=== Sepsis Outcomes ===")
if 'sepsis_event' in outcomes.columns:
    print(outcomes['sepsis_event'].value_counts())
    print(f"Sepsis rate: {outcomes['sepsis_event'].mean()*100:.2f}%")

if 'hospitalisation_required' in outcomes.columns:
    print(outcomes['hospitalisation_required'].value_counts())

if 'outcome_status' in outcomes.columns:
    print("\nOutcome Status:")
    print(outcomes['outcome_status'].value_counts())


=== Sepsis Outcomes ===
sepsis_event
False    528
True      72
Name: count, dtype: int64
Sepsis rate: 12.00%
hospitalisation_required
False    456
True     144
Name: count, dtype: int64

Outcome Status:
outcome_status
Recovered - discharged    529
Ongoing treatment          55
Transferred                10
Deceased                    6
Name: count, dtype: int64


### Merging Tables

In [54]:
# Starting with patients
full_data = patients.copy()

In [55]:
# Merge with vital signs (get latest vital signs per patient)
latest_vitals = vitals.sort_values('timestamp').groupby('patient_id').last().reset_index()
full_data = full_data.merge(latest_vitals, on='patient_id', how='left', suffixes=('', '_vitals'))

In [56]:
full_data.head()

,patient_id,age,gender,medical_conditions,registration_date,observation_id,timestamp,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure
0,1,66,Male,"Diabetes, Cancer (active)",2024-10-19,10,2024-10-22 15:20:00,90.9,36.65,98.7,15.2,115.2
1,2,42,Female,"Diabetes, Chronic Kidney Disease, Immunosuppre...",2025-06-24,32,2025-06-26 17:27:00,85.6,37.14,97.9,16.1,121.7
2,3,74,Male,Cancer (active),2024-02-28,47,2024-03-01 12:47:00,79.8,36.98,96.8,19.0,126.1
3,4,77,Female,"Diabetes, Hypertension",2024-07-31,63,2024-08-04 07:42:00,73.1,36.31,96.5,17.2,129.7
4,5,25,Female,NaN,2024-07-09,78,2024-07-10 18:06:00,69.3,37.05,97.6,11.4,136.4


In [57]:
# Merge with latest lab results
latest_labs = labs.sort_values('timestamp').groupby('patient_id').last().reset_index()
full_data = full_data.merge(latest_labs, on='patient_id', how='left', suffixes=('', '_lab'))

In [58]:
# Merge clinical history directly 
full_data = full_data.merge(history, on='patient_id', how='left', suffixes=('', '_history'))

In [59]:
# Merge with sepsis outcomes
full_data = full_data.merge(outcomes, on='patient_id', how='left')

print(f"Full dataset shape: {full_data.shape}")
print(f"Columns: {full_data.columns.tolist()}")

Full dataset shape: (1449, 29)
Columns: ['patient_id', 'age', 'gender', 'medical_conditions', 'registration_date', 'observation_id', 'timestamp', 'heart_rate', 'temperature', 'oxygen_saturation', 'respiratory_rate', 'blood_pressure', 'lab_id', 'timestamp_lab', 'white_cell_count', 'crp', 'lactate', 'creatinine', 'platelet_count', 'history_id', 'diagnosis_history', 'infection_history', 'medication_history', 'treatment_history', 'outcome_id', 'sepsis_event', 'diagnosis_time', 'hospitalisation_required', 'outcome_status']


In [60]:
full_data.head()

,patient_id,age,gender,medical_conditions,registration_date,observation_id,timestamp,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure,lab_id,timestamp_lab,white_cell_count,crp,lactate,creatinine,platelet_count,history_id,diagnosis_history,infection_history,medication_history,treatment_history,outcome_id,sepsis_event,diagnosis_time,hospitalisation_required,outcome_status
0,1,66,Male,"Diabetes, Cancer (active)",2024-10-19,10,2024-10-22 15:20:00,90.9,36.65,98.7,15.2,115.2,4,2024-10-22 01:59:00,6.70,12.0,0.55,0.68,327.0,1,Cancer (active),Skin/Soft Tissue,Atorvastatin,Oxygen therapy,1,False,NaT,True,Recovered - discharged
1,2,42,Female,"Diabetes, Chronic Kidney Disease, Immunosuppre...",2025-06-24,32,2025-06-26 17:27:00,85.6,37.14,97.9,16.1,121.7,7,2025-06-26 17:34:00,7.50,17.5,0.60,1.08,165.0,2,Immunosuppression,Respiratory,Insulin,IV fluids,2,False,NaT,False,Recovered - discharged
2,2,42,Female,"Diabetes, Chronic Kidney Disease, Immunosuppre...",2025-06-24,32,2025-06-26 17:27:00,85.6,37.14,97.9,16.1,121.7,7,2025-06-26 17:34:00,7.50,17.5,0.60,1.08,165.0,3,Chronic Kidney Disease,Surgical Site,Salbutamol inhaler,Surgery (recent),2,False,NaT,False,Recovered - discharged
3,3,74,Male,Cancer (active),2024-02-28,47,2024-03-01 12:47:00,79.8,36.98,96.8,19.0,126.1,10,2024-03-01 02:51:00,8.35,13.7,0.99,1.24,285.0,4,Cancer (active),Surgical Site,Atorvastatin,Oxygen therapy,3,False,NaT,True,Recovered - discharged
4,3,74,Male,Cancer (active),2024-02-28,47,2024-03-01 12:47:00,79.8,36.98,96.8,19.0,126.1,10,2024-03-01 02:51:00,8.35,13.7,0.99,1.24,285.0,5,Cancer (active),None documented,Amlodipine,NaN,3,False,NaT,True,Recovered - discharged


In [61]:
# Analyse vital signs trends
if 'timestamp' in vitals.columns:
    print("\n=== Vital Signs Timeline ===")
    
    # Get date range
    print(f"Date range: {vitals['timestamp'].min()} to {vitals['timestamp'].max()}")
    
    # Vital signs per patient over time
    vitals_per_patient = vitals.groupby('patient_id').size()
    print(f"\nAverage vitals records per patient: {vitals_per_patient.mean():.2f}")
    print(f"Max vitals records per patient: {vitals_per_patient.max()}")
    print(f"Min vitals records per patient: {vitals_per_patient.min()}")


=== Vital Signs Timeline ===
Date range: 2024-01-01 04:37:00 to 2025-12-29 07:13:00

Average vitals records per patient: 19.68
Max vitals records per patient: 30
Min vitals records per patient: 10


In [62]:
print("\n=== Correlation Analysis ===")
# Select numeric columns from merged data
numeric_cols = full_data.select_dtypes(include=['int64', 'float64']).columns
if len(numeric_cols) > 0:
    correlation_matrix = full_data[numeric_cols].corr()
    
    # Show correlations with sepsis_event
    if 'sepsis_event' in full_data.columns:
        sepsis_corr = correlation_matrix['sepsis_event'].sort_values(ascending=False)
        print("Top correlations with sepsis event:")
        print(sepsis_corr.head(10))


=== Correlation Analysis ===


KeyError: 'sepsis_event'

In [ ]:
 correlation_matrix.head()

,patient_id,age,observation_id,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure,lab_id,white_cell_count,crp,lactate,creatinine,platelet_count,outcome_id_x,history_id,outcome_id_y
patient_id,1.000000,-0.022669,0.999872,-0.007739,-0.010908,-0.004470,-0.028563,0.067407,0.999950,-0.036053,-0.034611,0.000383,-0.016242,0.029979,1.000000,0.999905,1.000000
age,-0.022669,1.000000,-0.021559,0.040332,0.016994,-0.038528,0.023693,-0.073004,-0.022234,0.037329,0.024039,0.055186,0.073240,0.055254,-0.022669,-0.022632,-0.022669
observation_id,0.999872,-0.021559,1.000000,-0.007635,-0.012776,-0.004087,-0.029528,0.067952,0.999873,-0.036335,-0.035487,-0.000412,-0.016069,0.029975,0.999872,0.999659,0.999872
heart_rate,-0.007739,0.040332,-0.007635,1.000000,0.333021,-0.572000,0.614374,-0.438781,-0.007590,0.333652,0.684635,0.602452,0.501762,-0.353320,-0.007739,-0.007194,-0.007739
temperature,-0.010908,0.016994,-0.012776,0.333021,1.000000,-0.336901,0.350387,-0.214147,-0.011035,0.192843,0.450061,0.402598,0.316525,-0.196143,-0.010908,-0.010074,-0.010908
